In [0]:
%pip install huggingface_hub[hf_xet]
%restart_python

  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.2 MB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import sys
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path
import numpy as np
from tqdm import tqdm
import warnings
import yaml
warnings.filterwarnings('ignore')

# Load configuration from config.yaml
config_path = Path('config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REWARD_MODELS = config['reward_models']

print("✓ Configuration loaded from config.yaml")
print(f"Available reward models: {list(REWARD_MODELS.keys())}")


2026-04-04 09:58:54.621484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775296734.638019   22370 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775296734.643146   22370 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775296734.656953   22370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775296734.656969   22370 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775296734.656971   22370 computation_placer.cc:177] computation placer alr

[2026-04-04 09:58:59,712] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


✓ PersonalLLM code imported successfully
Available reward models: ['llama3_sfairx', 'mistral_weqweasdas', 'mistral_ray', 'gemma_7b', 'mistral_raft', 'oasst_deberta_v3', 'oasst_pythia_1b', 'beaver_7b']


In [0]:
INPUT_PATH = "./data/generated_responses.parquet"
OUTPUT_PATH = "./data/scored_responses.parquet"

# Scoring configuration from config.yaml
MAX_LENGTH = config['scoring']['max_length']
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_parquet(INPUT_PATH)

print(f"   Loaded {len(df)} prompts × 2 response types = {len(df) * 2} responses to score")
print(f"   Divergence distribution: {df['divergence_type'].value_counts().to_dict()}")
print(f"Device: {DEVICE}")
print(f"Number of models to score: {len(REWARD_MODELS)}")

Device: cuda
Number of models to score: 8
Estimated runtime on L40s: ~40-55 minutes for all models


In [0]:
def format_prompt_response(prompt, response, tokenizer):
    """Format prompt and response using PersonalLLM's convention."""
    if tokenizer.chat_template is not None:
        try:
            formatted = tokenizer.apply_chat_template(
                [
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": response},
                ],
                tokenize=False,
                add_generation_prompt=False,
            )
            if tokenizer.bos_token:
                formatted = formatted.replace(tokenizer.bos_token, "")
            return formatted
        except Exception:
            pass
    return f"User: {prompt}\nAssistant: {response}"


def score_both_response_types(model_name, prompts, pref_responses, need_responses):
    """
    EFFICIENT: Load model once, score both response types, then unload.
    This cuts runtime in half compared to loading each model twice.
    """
    print(f"   Loading {model_name}...")

    try:
        from transformers import pipeline

        model_config = REWARD_MODELS[model_name]
        tokenizer = AutoTokenizer.from_pretrained(
            model_config['tokenizer_name'],
            trust_remote_code=True
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            model_config['model_name'],
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        )
        model.eval()

        # Set up padding
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if model.config.pad_token_id is None:
            model.config.pad_token_id = tokenizer.pad_token_id

        reward_pipe = pipeline(
            "text-classification",
            model=model,
            tokenizer=tokenizer,
            device=DEVICE
        )

        batch_size = 32
        pipe_kwargs = {
            "batch_size": batch_size,
            "truncation": True,
            "padding": "longest",
            "max_length": MAX_LENGTH,
            "function_to_apply": "none",
            "return_token_type_ids": False,
        }

        def score_batch(responses, desc):
            formatted = [format_prompt_response(p, r, tokenizer) for p, r in zip(prompts, responses)]
            scores = []
            for i in tqdm(range(0, len(formatted), batch_size), desc=f"     {desc}"):
                batch_texts = formatted[i:i+batch_size]
                outputs = reward_pipe(batch_texts, **pipe_kwargs)

                if isinstance(outputs[0], dict):
                    batch_scores = [o["score"] for o in outputs]
                elif isinstance(outputs[0][0], dict):
                    batch_scores = [o[0]["score"] for o in outputs]
                else:
                    batch_scores = [float(o[0].cpu().numpy()) if hasattr(o[0], 'cpu') else float(o[0]) for o in outputs]
                scores.extend(batch_scores)
            return scores

        # Score both types (model stays in memory - efficient!)
        print(f"     Scoring preference-matched...")
        pref_scores = score_batch(pref_responses, "Pref")

        print(f"     Scoring need-aware...")
        need_scores = score_batch(need_responses, "Need")

        # Cleanup
        del model, tokenizer, reward_pipe
        torch.cuda.empty_cache()

        print(f"     ✓ Pref: mean={np.mean(pref_scores):.3f}, Need: mean={np.mean(need_scores):.3f}")
        return pref_scores, need_scores

    except Exception as e:
        import traceback
        print(f"     ✗ Error: {e}")
        print(traceback.format_exc())
        return [0.0] * len(prompts), [0.0] * len(prompts)

In [0]:
print("="*80)
print(f"EFFICIENT SCORING: Each model loaded once for both response types")
print(f"Models: {len(REWARD_MODELS)} | Responses: {len(df)} × 2 = {len(df)*2}")
print("="*80 + "\n")

prompts = df['test_prompt'].tolist()
pref_responses = df['preference_matched_response'].tolist()
need_responses = df['need_aware_response'].tolist()

all_pref_scores = {}
all_need_scores = {}

for idx, model_name in enumerate(REWARD_MODELS, 1):
    print(f"📊 Model {idx}/{len(REWARD_MODELS)}: {model_name}")

    pref_scores, need_scores = score_both_response_types(
        model_name, prompts, pref_responses, need_responses
    )

    all_pref_scores[f'preference_reward_{model_name}'] = pref_scores
    all_need_scores[f'need_aware_reward_{model_name}'] = need_scores
    print()

print("="*80)
print("✓ All models scored!")
print("="*80)

In [0]:
# Compute means
print("Computing mean scores across all models...")

pref_cols = list(all_pref_scores.keys())
all_pref_scores['preference_reward_mean'] = [
    np.mean([all_pref_scores[col][i] for col in pref_cols])
    for i in range(len(prompts))
]

need_cols = list(all_need_scores.keys())
all_need_scores['need_aware_reward_mean'] = [
    np.mean([all_need_scores[col][i] for col in need_cols])
    for i in range(len(prompts))
]

# Create DataFrames
pref_scores_df = pd.DataFrame(all_pref_scores)
need_scores_df = pd.DataFrame(all_need_scores)

# Combine with original data
result_df = pd.concat([df, pref_scores_df, need_scores_df], axis=1)
result_df['reward_gap'] = result_df['preference_reward_mean'] - result_df['need_aware_reward_mean']

# Save
result_df.to_parquet(OUTPUT_PATH, index=False)
pref_scores_df.to_parquet("./data/pref_scores.parquet")
need_scores_df.to_parquet("./data/need_scores.parquet")

# Summary statistics
print("\n" + "="*80)
print("SCORING SUMMARY")
print("="*80)
print(f"\nTotal responses: {len(result_df) * 2}")
print(f"Models used: {len(REWARD_MODELS)}")

print(f"\nPreference-matched rewards:")
print(f"  Mean: {result_df['preference_reward_mean'].mean():.4f}")
print(f"  Std:  {result_df['preference_reward_mean'].std():.4f}")

print(f"\nNeed-aware rewards:")
print(f"  Mean: {result_df['need_aware_reward_mean'].mean():.4f}")
print(f"  Std:  {result_df['need_aware_reward_mean'].std():.4f}")

print(f"\nReward gap (pref - need):")
print(f"  Mean: {result_df['reward_gap'].mean():.4f}")
print(f"  Std:  {result_df['reward_gap'].std():.4f}")

pref_wins = (result_df['preference_reward_mean'] > result_df['need_aware_reward_mean']).sum()
print(f"\nPreference-matched scores higher: {pref_wins}/{len(result_df)} ({100*pref_wins/len(result_df):.1f}%)")

print(f"\nBreakdown by divergence type:")
for div_type in sorted(result_df['divergence_type'].unique()):
    subset = result_df[result_df['divergence_type'] == div_type]
    print(f"\n  {div_type} (n={len(subset)}):")
    print(f"    Pref: {subset['preference_reward_mean'].mean():.4f}")
    print(f"    Need: {subset['need_aware_reward_mean'].mean():.4f}")
    print(f"    Gap:  {subset['reward_gap'].mean():+.4f}")

print(f"\n✓ Results saved to: {OUTPUT_PATH}")
print("="*80)


Scoring preference responses

📊 Model 1/8: llama3_sfairx
   Loading llama3_sfairx...
/local_disk0/llama3_sfairx
Cache directory: /local_disk0


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model downloaded successfully to: /local_disk0/models--sfairXC--FsfairX-LLaMA3-RM-v0.1/snapshots/09c5e34065150c51ddb6a3078505cab130f9b08b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda


...

(…)5eb1ba5/model-00001-of-00007.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

(…)5eb1ba5/model-00002-of-00007.safetensors:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

(…)2a1cfe5b758300ddda5eb1ba5/tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

(…)b758300ddda5eb1ba5/tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

Model downloaded successfully to: /local_disk0/models--PKU-Alignment--beaver-7b-v1.0-cost/snapshots/0aa7f58040a83a92a1cfe5b758300ddda5eb1ba5
     ✗ Error with beaver_7b: Unrecognized model in /local_disk0/beaver_7b. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, d